# 🤖 Project 6.1 — Robot Arm Configuration Tree
**DSA for Mechatronics · Week 06 — Trees & BSTs**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A 6-DOF industrial robot arm stores its workspace configurations in a **binary tree**.
Each node holds a joint-angle snapshot (a configuration ID and a "reachability score").
The control software must traverse this tree in different orders depending on the task:
- **Preorder** — broadcast configuration ID before children (serialise for network)
- **Inorder** — visit in sorted score order (rank configurations)
- **Postorder** — process children before parent (free memory bottom-up)
- **Level-order** — scan row by row (visualise the workspace)

Your tree is generated deterministically from your student ID.


## Step 1 — Build your binary tree

In [ ]:
from collections import deque

class TNode:
    """One configuration snapshot in the robot workspace tree."""
    def __init__(self, cfg_id, score):
        self.cfg_id = cfg_id   # unique configuration identifier
        self.score  = score    # reachability score 0–100
        self.left   = None
        self.right  = None
    def __repr__(self):
        return f"Cfg({self.cfg_id}, s={self.score})"

# ── Personalised parameters ────────────────────────────────────────
N_NODES      = random.randint(12, 20)
SCORE_MIN    = random.randint(10, 30)
SCORE_MAX    = random.randint(70, 100)
CFG_ID_START = random.randint(100, 500)

# Build a complete-ish binary tree from a random list (level-order insertion)
cfg_ids = random.sample(range(CFG_ID_START, CFG_ID_START + N_NODES * 3), N_NODES)
scores  = [random.randint(SCORE_MIN, SCORE_MAX) for _ in range(N_NODES)]

def build_tree(ids, scores):
    """Insert nodes level-by-level (complete binary tree shape)."""
    if not ids: return None
    nodes = [TNode(i, s) for i, s in zip(ids, scores)]
    for i in range(1, len(nodes)):
        parent = nodes[(i - 1) // 2]
        if i % 2 == 1:
            parent.left  = nodes[i]
        else:
            parent.right = nodes[i]
    return nodes[0]

root = build_tree(cfg_ids, scores)

def tree_size(node):
    if not node: return 0
    return 1 + tree_size(node.left) + tree_size(node.right)

def tree_height(node):
    if not node: return 0
    return 1 + max(tree_height(node.left), tree_height(node.right))

print(f"Tree parameters:")
print(f"  Nodes          : {N_NODES}")
print(f"  Score range    : {SCORE_MIN} – {SCORE_MAX}")
print(f"  Tree height    : {tree_height(root)}")
print(f"  Root config    : {root}")
print()
# Print level-order preview
print("  Level-order preview:")
q, level = deque([root]), 0
while q:
    width = len(q)
    row   = []
    for _ in range(width):
        n = q.popleft()
        row.append(f"Cfg{n.cfg_id}(s={n.score})")
        if n.left:  q.append(n.left)
        if n.right: q.append(n.right)
    print(f"  L{level}: {' | '.join(row)}")
    level += 1


## Step 2 — All four traversals

In [ ]:
# ── Recursive traversals ────────────────────────────────────────────
def preorder(node, result=None):
    """Root → Left → Right  (used for serialisation / copy)"""
    if result is None: result = []
    if not node: return result
    result.append(node.cfg_id)
    preorder(node.left,  result)
    preorder(node.right, result)
    return result

def inorder(node, result=None):
    """Left → Root → Right  (visits BST nodes in sorted order)"""
    if result is None: result = []
    if not node: return result
    inorder(node.left,  result)
    result.append(node.cfg_id)
    inorder(node.right, result)
    return result

def postorder(node, result=None):
    """Left → Right → Root  (process children before parent)"""
    if result is None: result = []
    if not node: return result
    postorder(node.left,  result)
    postorder(node.right, result)
    result.append(node.cfg_id)
    return result

def level_order(root):
    """BFS level-by-level using a queue."""
    if not root: return []
    result, q = [], deque([root])
    while q:
        node = q.popleft()
        result.append(node.cfg_id)
        if node.left:  q.append(node.left)
        if node.right: q.append(node.right)
    return result

pre  = preorder(root)
ino  = inorder(root)
post = postorder(root)
lvl  = level_order(root)

print("Traversal results (configuration IDs):")
print(f"  Preorder   : {pre}")
print(f"  Inorder    : {ino}")
print(f"  Postorder  : {post}")
print(f"  Level-order: {lvl}")
print()
print(f"  First config visited (preorder)  : {pre[0]}")
print(f"  Last config freed  (postorder)   : {post[-1]}")
print(f"  Root in level-order              : {lvl[0]}")


## Step 3 — Tree statistics: height, diameter, leaf count

In [ ]:
def count_leaves(node):
    """Count nodes with no children."""
    if not node: return 0
    if not node.left and not node.right: return 1
    return count_leaves(node.left) + count_leaves(node.right)

def tree_diameter(node):
    """
    Diameter = longest path between any two nodes (in edges).
    At each node, the longest path through it = left_height + right_height.
    We compute height and update diameter simultaneously — O(n).
    """
    best = [0]
    def height(n):
        if not n: return 0
        lh = height(n.left)
        rh = height(n.right)
        best[0] = max(best[0], lh + rh)
        return 1 + max(lh, rh)
    height(node)
    return best[0]

def max_score_path(node):
    """
    Find the root-to-leaf path with the highest total score sum.
    Returns (max_sum, path_of_cfg_ids).
    """
    if not node: return (0, [])
    if not node.left and not node.right:
        return (node.score, [node.cfg_id])
    best = (-1, [])
    for child in (node.left, node.right):
        if child:
            s, path = max_score_path(child)
            if s > best[0]:
                best = (s, path)
    total = node.score + best[0]
    return (total, [node.cfg_id] + best[1])

height   = tree_height(root)
diameter = tree_diameter(root)
n_leaves = count_leaves(root)
max_sum, best_path = max_score_path(root)

print("Tree statistics:")
print(f"  Total nodes  : {N_NODES}")
print(f"  Height       : {height}")
print(f"  Diameter     : {diameter} edges")
print(f"  Leaf nodes   : {n_leaves}")
print(f"  Internal nodes: {N_NODES - n_leaves}")
print()
print(f"Best root-to-leaf path (highest score sum):")
print(f"  Path  : {' → '.join(str(c) for c in best_path)}")
print(f"  Score sum: {max_sum}")


## Step 4 — Level statistics: average score per level

In [ ]:
def level_averages(root):
    """
    Compute the average reachability score at each tree level.
    BFS with level-tracking using a queue.
    Returns list of (level, avg_score, node_count).
    """
    if not root: return []
    results = []
    q = deque([root])
    level = 0
    while q:
        width  = len(q)
        scores_at_level = []
        for _ in range(width):
            n = q.popleft()
            scores_at_level.append(n.score)
            if n.left:  q.append(n.left)
            if n.right: q.append(n.right)
        avg = round(sum(scores_at_level) / len(scores_at_level), 1)
        results.append((level, avg, len(scores_at_level)))
        level += 1
    return results

level_stats = level_averages(root)
best_level  = max(level_stats, key=lambda x: x[1])

print("Level-by-level score averages:")
print(f"  {'Level':>6}  {'Nodes':>6}  {'Avg score':>10}  Bar")
print(f"  {'─'*6}  {'─'*6}  {'─'*10}  {'─'*20}")
for lv, avg, cnt in level_stats:
    bar = "█" * int(avg / 5)
    print(f"  {lv:6d}  {cnt:6d}  {avg:10.1f}  {bar}")

print(f"\n  Best level : Level {best_level[0]} (avg score {best_level[1]})")


## 📋 Final Report

In [ ]:
W = 56
print("╔" + "═"*W + "╗")
print("║" + " ROBOT ARM CONFIGURATION TREE — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<26}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<26}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  TREE PARAMETERS" + " "*(W-17) + "║")
print(f"║  {'Total nodes':<26}: {N_NODES:<26}║")
print(f"║  {'Score range':<26}: {SCORE_MIN} – {SCORE_MAX}{'':<22}║")
print(f"║  {'Height':<26}: {height:<26}║")
print(f"║  {'Diameter':<26}: {diameter} edges{'':<20}║")
print(f"║  {'Leaf nodes':<26}: {n_leaves:<26}║")
print("╠" + "═"*W + "╣")
print("║  TRAVERSALS (first 6 IDs shown)" + " "*(W-32) + "║")
print(f"║  {'Preorder start':<26}: {str(pre[:6]):<26}║")
print(f"║  {'Inorder start':<26}: {str(ino[:6]):<26}║")
print(f"║  {'Postorder start':<26}: {str(post[:6]):<26}║")
print(f"║  {'Level-order start':<26}: {str(lvl[:6]):<26}║")
print("╠" + "═"*W + "╣")
print("║  ANALYSIS" + " "*(W-10) + "║")
print(f"║  {'Best level (avg score)':<26}: Level {best_level[0]} ({best_level[1]}){'':<14}║")
print(f"║  {'Best root-to-leaf sum':<26}: {max_sum} via Cfg{best_path[0]}{'':<12}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about the system?

*Your answer here:*

---

**Q2 — Which tree concept did you find most important, and why?**
Pick one technique from the notebook (traversal, BST property, recursion pattern…) and explain in your own words what problem it solves.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
